# blendfleet — Persistent Multi-GPU Blender Worker

Connects this Google Colab / Kaggle GPU (`T4`, `L4`, `G4`) to a `blendfleet` render job.
- Loads `.blend` **once** per worker (zero per-frame restart overhead).
- Auto-selects **OptiX** on `T4` / `L4` / `G4` (hardware RT cores) and **CUDA** on `A100` / `H100`.
- Preloads `libtcmalloc_minimal` to prevent Colab glibc OOM crashes.
- Automatically recovers frames if another worker tab disconnects.


In [ ]:
#@title 1. Configure blendfleet Worker { display-mode: 'form' }
MODE = "http" #@param ["http", "fs"]
COORDINATOR_URL = "" #@param {type:"string"}
TOKEN = "" #@param {type:"string"}
BLENDER_VERSION = "5.2.3" #@param ["4.2.23", "4.5.14", "5.2.2", "5.2.3"] {allow-input: true}
DEVICE = "auto" #@param ["auto", "optix", "cuda", "cpu"]
USE_DRIVE_CACHE = False #@param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/blendfleet_cache" #@param {type:"string"}
#@markdown ---
#@markdown ### Shared Google Drive Mode Only (`MODE = 'fs'`)
SHARED_SCENE_PATH = "/content/drive/MyDrive/blendfleet_job/scene.blend" #@param {type:"string"}
SHARED_OUTPUT_DIR = "/content/drive/MyDrive/blendfleet_job/frames" #@param {type:"string"}
FRAMES = "1-240" #@param {type:"string"}
ENGINE = "CYCLES" #@param ["CYCLES", "EEVEE"]
SAMPLES = 0 #@param {type:"integer"}
WORKER_INDEX = 0 #@param {type:"integer"}
TOTAL_WORKERS = 1 #@param {type:"integer"}


In [ ]:
#@title 2. Check GPU, Install tcmalloc & blendfleet
import os, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'CPU runtime'
!apt-get -qq update && apt-get -qq install -y libtcmalloc-minimal4 >/dev/null 2>&1 || true
!pip -q install --upgrade git+https://github.com/brixtonpham/blendfleet.git
if USE_DRIVE_CACHE or MODE == 'fs':
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
#@title 3. Launch Persistent Blender Work-Stealing Loop
import shlex, subprocess
cache_dir = DRIVE_CACHE_DIR if (USE_DRIVE_CACHE or MODE == 'fs') else '/content/blendfleet_cache'
if MODE == 'http':
    assert COORDINATOR_URL and TOKEN, 'Set COORDINATOR_URL and TOKEN in Cell 1.'
    cmd = [
        'blendfleet', 'worker', '--mode', 'http',
        '--coordinator-url', COORDINATOR_URL,
        '--token', TOKEN,
        '--blender-version', BLENDER_VERSION,
        '--cache-dir', cache_dir,
        '--device', DEVICE,
    ]
else:
    cmd = [
        'blendfleet', 'worker', '--mode', 'fs',
        '--scene', SHARED_SCENE_PATH,
        '--out', SHARED_OUTPUT_DIR,
        '--frames', FRAMES,
        '--engine', ENGINE,
        '--device', DEVICE,
        '--blender-version', BLENDER_VERSION,
        '--cache-dir', cache_dir,
        '--worker-index', str(WORKER_INDEX),
        '--total-workers', str(TOTAL_WORKERS),
    ]
    if SAMPLES > 0:
        cmd.extend(['--samples', str(SAMPLES)])
print('Running:', shlex.join(cmd))
subprocess.run(cmd, check=True)
